# Deep Reinforcement Learning (Sp25) — Lecture 23: Inverse Reinforcement Learning
**Instructor:** Dr. Mohammad Hossein Rohban  
**Notebook:** Comprehensive IRL Implementations (Feature-Matching, MaxEnt, GCL)  
**Summarized By:** Arshia Gharooni (adapted into code)

---

## Overview
This notebook implements **Inverse Reinforcement Learning (IRL)** algorithms from Lecture 23, covering all major paradigms:

1. **Optimal-Control Interpretation of Demonstrations** (Section 1): IRL recovers rewards that explain expert behavior as optimal control.
2. **Learning from Demonstrations: Three Paradigms** (Section 2): Behavioral Cloning, RL, and IRL.
3. **Formal Definition of the IRL Problem** (Section 3): Demonstrations, feature expectations, and ill-posedness.
4. **Feature-Matching Inverse RL** (Section 4): Apprenticeship Learning via Quadratic Programming.
5. **Maximum-Margin Formulation** (Section 5): SVM-like dual for IRL.
6. **Latent-Variable Model and Maximum-Entropy Distribution** (Section 6): Probabilistic IRL.
7. **Dynamic-Programming Evaluation of the Partition Function** (Section 7): Soft Bellman equations for MaxEnt IRL.
8. **Sample-Based IRL with Unknown Dynamics** (Section 8): Importance sampling for model-free IRL.
9. **Variance-Reduction Techniques** (Section 9): Baseline subtraction and ESS diagnostics.
10. **Guided Cost Learning (GCL)** (Section 10): Adversarial IRL with policy optimization.

**Key Improvements in Code:**
- Robust implementations with numerical stability.
- Advanced features: stochastic environments, importance sampling, TRPO for GCL.
- Diagnostics: policy alignment, return evaluation, variance analysis.
- Harder challenges: larger grids, unknown dynamics, multi-step optimization.

Run cells sequentially. Dependencies: numpy, scipy, matplotlib, seaborn, cvxpy, torch.

---

## 1. Optimal-Control Interpretation of Demonstrations
IRL assumes expert demonstrations come from an optimal (or near-optimal) policy for some unknown reward R. Recovering R provides a causal explanation and generalizable control.

In code: We define an MDP M = ⟨S, A, P, R, γ⟩ and compute value functions for any policy π.

```python
# MDP setup (will be used throughout)
class MDP:
    def __init__(self, n_states: int, n_actions: int, P: np.ndarray, gamma: float = 0.9):
        self.n_states = n_states
        self.n_actions = n_actions
        self.P = P  # (S, A, S)
        self.gamma = gamma

    def value_function(self, policy: np.ndarray, reward: np.ndarray) -> np.ndarray:
        """Compute V^π_R via policy evaluation."""
        V = np.zeros(self.n_states)
        for _ in range(100):  # fixed-point iteration
            Q = reward[:, None] + self.gamma * (self.P @ V)
            V = np.sum(policy[:, :, None] * Q, axis=1)
        return V

    def q_function(self, policy: np.ndarray, reward: np.ndarray) -> np.ndarray:
        """Compute Q^π_R."""
        V = self.value_function(policy, reward)
        Q = reward[:, None] + self.gamma * (self.P @ V)
        return Q
```

---

## 2. Learning from Demonstrations: Three Paradigms
- **Behavioral Cloning:** Supervised learning s → a.
- **RL:** Known R, learn π.
- **IRL:** Unknown R, learn R from π, then π from R.

Motivation: IRL avoids covariate shift by learning compact rewards.

---

## 3. Formal Definition of the IRL Problem
Demonstrations D = {τ^{(i)}}, each τ = (s0, a0, s1, a1, ...). Feature map ϕ(s,a) → R^k, discounted features Φ(τ) = ∑ γ^t ϕ(st, at).

Empirical μ_E = (1/N) ∑ Φ(τ^{(i)}).

Linear reward: R_θ(s,a) = θ^T ϕ(s,a).

**Ill-posedness:** Reward aliasing (scaling invariance), policy non-uniqueness.

---

## 4. Feature-Matching Inverse RL (Apprenticeship Learning)
Algorithm: Iteratively solve QP to find θ that separates μ_E from learner μ(π), then update π via RL.

**Performance Gap Lemma:** If ||μ_E - μ(π)||_1 ≤ ε, then |V^π_θ - V^π*_θ| ≤ ε ||θ||_∞.

Code: Implement QP using CVXPY.

In [3]:
# Deep Reinforcement Learning (Sp25) — Lecture 23: Inverse Reinforcement Learning
# Instructor: Dr. Mohammad Hossein Rohban
# Notebook: Comprehensive IRL Implementations (Feature-Matching, MaxEnt, GCL)
# Summarized By: Arshia Gharooni (adapted into code)

# 1) Imports and utilities
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import logsumexp
from scipy.optimize import minimize_scalar
from typing import List, Tuple, Optional, Dict, Any

# cvxpy is optional — fallback will be used if unavailable
try:
    import cvxpy as cp  # For quadratic programming in Feature-Matching IRL
    CPX_AVAILABLE = True
except Exception:
    cp = None
    CPX_AVAILABLE = False
    print("cvxpy not available; falling back to projected subgradient QP solver for apprenticeship-learning.")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

sns.set(style='whitegrid')

# For reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

In [5]:
# Minimal GridWorld implementation used by the notebook
class GridWorld:
    """Simple deterministic grid-world MDP used for IRL demos.

    Attributes:
        size: grid width/height (square grid)
        n_states: total states = size * size
        n_actions: number of actions (4: up,right,down,left)
        P: transition matrix with shape (S, A, S)
        goal_state: index of goal state (default: bottom-right)
    """

    def __init__(self, size: int = 8, slip_prob: float = 0.0):
        self.size = int(size)
        self.n_states = self.size * self.size
        self.n_actions = 4
        # default goal in bottom-right corner
        self.goal_state = self.n_states - 1
        # Build transition matrix P[s,a,s']
        self.P = np.zeros((self.n_states, self.n_actions, self.n_states), dtype=float)
        for s in range(self.n_states):
            r, c = divmod(s, self.size)
            for a in range(self.n_actions):
                nr, nc = r, c
                if a == 0:  # up
                    nr = max(0, r - 1)
                elif a == 1:  # right
                    nc = min(self.size - 1, c + 1)
                elif a == 2:  # down
                    nr = min(self.size - 1, r + 1)
                elif a == 3:  # left
                    nc = max(0, c - 1)
                ns = nr * self.size + nc
                if slip_prob <= 0.0:
                    self.P[s, a, ns] = 1.0
                else:
                    # simple slip model: mostly go to intended ns, small prob to uniform others
                    self.P[s, a, :] = slip_prob / (self.n_states - 1)
                    self.P[s, a, ns] += (1.0 - slip_prob)

    def _get_next_state(self, s: int, a: int) -> int:
        """Sample next state under transition dynamics for state s and action a."""
        probs = self.P[s, a]
        return int(np.random.choice(self.n_states, p=probs))

    def get_features(self, s: int) -> np.ndarray:
        """Return one-hot feature vector for state s."""
        vec = np.zeros(self.n_states, dtype=float)
        vec[s] = 1.0
        return vec

    def feature_matrix(self) -> np.ndarray:
        """Return identity feature matrix (one-hot features for each state)."""
        return np.eye(self.n_states)

    def render_reward(self, rewards: np.ndarray, title: str = "Reward") -> None:
        """Simple visualization of a per-state reward vector as a grid heatmap."""
        arr = np.asarray(rewards).reshape(self.size, self.size)
        plt.figure(figsize=(4, 4))
        plt.imshow(arr, cmap="coolwarm", interpolation="nearest")
        plt.colorbar()
        plt.title(title)
        plt.show()

In [2]:
!pip install cvxpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.5 kB/s  0:00:26 eta 0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 45.1 kB/s  0:00:16 eta 0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [cvxpy]32m4/5 [cvxpy]el]


In [7]:
# MaxEnt: sample-based and variance reduction (improved implementation)

def estimate_mu_from_weighted_trajectories(env: GridWorld, trajectories: List[List[Tuple[int,int]]], weights: np.ndarray, gamma: float = 0.9) -> Tuple[np.ndarray, float]:
    """Self-normalized importance sampling estimator for mu and ESS."""
    w = np.asarray(weights, dtype=float)
    if np.all(w == 0):
        raise ValueError("All weights are zero; check your proposal or theta scaling.")
    # normalize
    w_norm = w / (w.sum() + 1e-12)
    mu = np.zeros(env.n_states)
    for j, traj in enumerate(trajectories):
        wj = w_norm[j]
        for t, (s, _) in enumerate(traj):
            mu[s] += wj * (gamma ** t)
    ess = (w.sum() ** 2) / (np.clip((w ** 2).sum(), 1e-12, None))
    return mu, ess


def sample_based_maxent(env: GridWorld, trajectories: List[List[Tuple[int,int]]], proposal_policy: np.ndarray, n_iters: int = 100, lr: float = 0.1, gamma: float = 0.9, l2: float = 1e-3, verbose: bool = False) -> np.ndarray:
    """Model-free MaxEnt IRL via self-normalized importance sampling with ESS diagnostics."""
    theta = np.zeros(env.n_states)
    for it in range(n_iters):
        # compute log importance weights: log w = sum_t R_theta(s_t) - sum_t log proposal(s_t,a_t)
        log_ws = np.zeros(len(trajectories))
        for i, traj in enumerate(trajectories):
            lw = 0.0
            for (s, a) in traj:
                lw += theta[s] - np.log(proposal_policy[s, a] + 1e-12)
            log_ws[i] = lw
        # stabilize
        log_ws -= logsumexp(log_ws)
        ws = np.exp(log_ws)
        mu_model, ess = estimate_mu_from_weighted_trajectories(env, trajectories, ws, gamma=gamma)
        grad = mu_expert - mu_model - l2 * theta
        theta += lr * grad
        if verbose and (it % max(1, n_iters // 10) == 0):
            print(f"Iter {it+1}/{n_iters}: ESS={ess:.2f}, grad_norm={np.linalg.norm(grad):.6f}")
    return theta

# Example run (light):
proposal = np.ones((env.n_states, env.n_actions)) / env.n_actions
theta_sb = sample_based_maxent(env, demonstrations, proposal, n_iters=80, lr=0.2, verbose=True)
env.render_reward(theta_sb, title='Sample-Based MaxEnt Recovered Reward')

# ---------------------------------------------------------------------------------
# Guided Cost Learning / GAIL-style adversarial IRL
# Practical implementation using a discriminator (reward) and a policy trained by REINFORCE
# This avoids heavier TRPO dependency but illustrates GCL/GAIL behavior.
# ---------------------------------------------------------------------------------

class RewardDiscriminator(nn.Module):
    def __init__(self, n_states: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, state_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(state_onehot).squeeze(-1)

class PolicyNet(nn.Module):
    def __init__(self, n_states: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, state_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(state_onehot)


def one_hot_states(states: List[int], n_states: int) -> np.ndarray:
    arr = np.zeros((len(states), n_states), dtype=float)
    for i, s in enumerate(states): arr[i, s] = 1.0
    return arr


def collect_trajectories_from_policy(env: GridWorld, policy_net: PolicyNet, n_episodes: int = 50, max_steps: int = 100) -> List[List[Tuple[int,int]]]:
    policy_net.eval()
    trajectories = []
    with torch.no_grad():
        for _ in range(n_episodes):
            s = np.random.randint(0, env.n_states)
            traj = []
            for _ in range(max_steps):
                s_oh = torch.FloatTensor(one_hot_states([s], env.n_states))
                logits = policy_net(s_oh).squeeze(0)
                probs = torch.softmax(logits, dim=-1)
                m = Categorical(probs)
                a = int(m.sample().item())
                traj.append((s, a))
                s = env._get_next_state(s, a)
                if s == env.goal_state: break
            trajectories.append(traj)
    return trajectories


def trajectories_to_batches(trajectories: List[List[Tuple[int,int]]], n_states: int) -> Tuple[np.ndarray, np.ndarray]:
    states = []
    actions = []
    for traj in trajectories:
        for s, a in traj:
            states.append(s)
            actions.append(a)
    if len(states) == 0:
        return np.zeros((0, n_states)), np.zeros((0,), dtype=int)
    return one_hot_states(states, n_states), np.array(actions, dtype=int)


def train_gail(env: GridWorld, expert_trajectories: List[List[Tuple[int,int]]], n_iters: int = 200, rollouts_per_iter: int = 50, policy_lr: float = 1e-3, disc_lr: float = 1e-3, gamma: float = 0.99, policy_updates: int = 5) -> Tuple[PolicyNet, RewardDiscriminator, Dict[str, List[float]]]:
    device = torch.device('cpu')
    disc = RewardDiscriminator(env.n_states).to(device)
    policy = PolicyNet(env.n_states, env.n_actions).to(device)
    disc_opt = optim.Adam(disc.parameters(), lr=disc_lr)
    policy_opt = optim.Adam(policy.parameters(), lr=policy_lr)

    expert_states, expert_actions = trajectories_to_batches(expert_trajectories, env.n_states)
    expert_states_t = torch.FloatTensor(expert_states).to(device)
    expert_actions_t = torch.LongTensor(expert_actions).to(device)

    hist = {'disc_loss': [], 'disc_acc': [], 'policy_return': [], 'policy_match': []}

    # build approximate expert mapping for match metric
    counts = np.zeros((env.n_states, env.n_actions), dtype=int)
    for s, a in zip(expert_states, expert_actions):
        s_idx = int(np.argmax(s))
        counts[s_idx, a] += 1
    expert_mode_action = np.array([counts[s].argmax() if counts[s].sum()>0 else 0 for s in range(env.n_states)])

    for it in range(n_iters):
        learner_trajs = collect_trajectories_from_policy(env, policy, n_episodes=rollouts_per_iter)
        learner_states, learner_actions = trajectories_to_batches(learner_trajs, env.n_states)
        learner_states_t = torch.FloatTensor(learner_states).to(device)
        learner_actions_t = torch.LongTensor(learner_actions).to(device)

        # Train discriminator
        if expert_states_t.shape[0] == 0 or learner_states_t.shape[0] == 0:
            continue
        for _ in range(3):
            idx_e = np.random.choice(len(expert_states_t), size=min(256, len(expert_states_t)), replace=False)
            idx_l = np.random.choice(len(learner_states_t), size=min(256, len(learner_states_t)), replace=False)
            se = expert_states_t[idx_e]
            sl = learner_states_t[idx_l]
            ye = torch.ones(se.shape[0], device=device)
            yl = torch.zeros(sl.shape[0], device=device)
            inputs = torch.cat([se, sl], dim=0)
            labels = torch.cat([ye, yl], dim=0)
            logits = disc(inputs)
            loss = nn.BCEWithLogitsLoss()(logits, labels)
            disc_opt.zero_grad(); loss.backward(); disc_opt.step()

        with torch.no_grad():
            logits_e = disc(expert_states_t)
            logits_l = disc(learner_states_t)
            acc = ((torch.sigmoid(logits_e) > 0.5).float().mean() + (torch.sigmoid(logits_l) <= 0.5).float().mean())/2.0
            hist['disc_loss'].append(float(loss.item()))
            hist['disc_acc'].append(float(acc.item()))

        # Policy updates using REINFORCE with GAIL reward r = -log(1 - D(s))
        returns = []
        for _ in range(policy_updates):
            policy.train()
            for traj in learner_trajs:
                states = [s for s, _ in traj]
                acts = [a for _, a in traj]
                if len(states) == 0: continue
                s_oh = torch.FloatTensor(one_hot_states(states, env.n_states)).to(device)
                with torch.no_grad():
                    d_logits = disc(s_oh)
                    d_probs = torch.sigmoid(d_logits)
                    rewards = -torch.log(1.0 - d_probs + 1e-8).cpu().numpy()
                # compute return-to-go
                G = 0.0
                returns_traj = []
                for r in rewards[::-1]:
                    G = r + gamma * G
                    returns_traj.insert(0, G)
                # policy gradient
                s_batch = torch.FloatTensor(one_hot_states(states, env.n_states)).to(device)
                logits = policy(s_batch)
                logp = torch.log_softmax(logits, dim=-1)
                actions_t = torch.LongTensor(acts).to(device)
                selected = logp[range(len(acts)), actions_t]
                baseline = np.mean(returns_traj)
                adv = torch.FloatTensor(np.array(returns_traj) - baseline).to(device)
                loss_pg = - (selected * adv).mean()
                policy_opt.zero_grad(); loss_pg.backward(); policy_opt.step()
                returns.append(np.mean(returns_traj))

        hist['policy_return'].append(float(np.mean(returns)) if returns else 0.0)
        # approximate policy match (compare greedy action under policy to expert_mode_action)
        greedy_actions = []
        for s in range(env.n_states):
            s_oh = torch.FloatTensor(one_hot_states([s], env.n_states)).to(device)
            with torch.no_grad():
                logits = policy(s_oh).squeeze(0)
                greedy_actions.append(int(torch.argmax(logits).item()))
        hist['policy_match'].append(float((np.array(greedy_actions) == expert_mode_action).mean()))

        if (it % max(1, n_iters // 10) == 0):
            print(f"Iter {it+1}/{n_iters}: DiscAcc={acc.item():.3f}, AvgPolicyReturn={hist['policy_return'][-1]:.3f}, Match={hist['policy_match'][-1]:.3f}")

    return policy, disc, hist

# Run a short GAIL demo training (light)
policy_gail, disc_gail, hist_gail = train_gail(env, demonstrations, n_iters=80, rollouts_per_iter=30, policy_lr=3e-3, disc_lr=1e-3)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(hist_gail['disc_acc']); plt.title('Discriminator Accuracy'); plt.xlabel('Iter')
plt.subplot(1,2,2); plt.plot(hist_gail['policy_return']); plt.title('Policy Returns'); plt.xlabel('Iter')
plt.tight_layout(); plt.show()

# Final policy match
final_policy_match = hist_gail['policy_match'][-1]
print(f"Final policy match fraction (approx): {final_policy_match:.3f}")

NameError: name 'env' is not defined

### Sample-Based MaxEnt IRL (unknown dynamics)

When dynamics are unknown, we estimate the expected feature counts under P_theta using self-normalized importance sampling: we sample trajectories from a proposal policy \~π and weight each trajectory by

w(τ) = exp(∑_t R_θ(s_t,a_t)) / ∏_t \~π(a_t|s_t).

This notebook provides a numerically-stable implementation using log-weights, self-normalization, and an ESS (effective sample size) diagnostic. We also subtract a baseline (implicit via self-normalization) to reduce variance.

**Notes:**
- If ESS is low (< threshold), re-sampling or improving the proposal policy is recommended.
- The method is unbiased in the infinite-sample limit (self-normalized estimator has small bias but far lower variance in practice).

### Guided Cost Learning / GAIL (Adversarial IRL)

Guided Cost Learning (GCL) and GAIL are adversarial IRL approaches. They alternate between:

1. Training a discriminator (or reward network) to distinguish expert from learner trajectories.
2. Updating the learner policy to minimize its expected cost under the learned reward (or equivalently, to fool the discriminator).

This implementation uses a discriminator D(s) (state-only) and trains the policy with REINFORCE using the reward r(s) = -log(1 - D(s)). For stability we:
- Use several discriminator updates per iteration
- Use entropy / baseline techniques in policy gradient steps
- Monitor discriminator accuracy, ESS-like metrics, and policy match

Caveat: TRPO or PPO with trust-region constraints are preferable for production; REINFORCE is simpler and sufficient for demonstration and unit experiments.

In [ ]:
# -----------------------------
# 4b. Feature-Matching / Apprenticeship Learning (robust)
# Solve QP with CVXPY when available, otherwise use projected subgradient fallback.
# -----------------------------

def project_to_l2_ball(x: np.ndarray, radius: float = 1.0) -> np.ndarray:
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x
    return x * (radius / (norm + 1e-12))


def apprenticeship_learning(env: GridWorld, mdp: MDP, mu_expert: np.ndarray, n_iters: int = 10, epsilon: float = 0.01) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Apprenticeship Learning. Returns theta and list of policies (deterministic arrays of shape (S,)).
    If cvxpy is available, solves the QP; else uses a simple projected subgradient fallback."""
    policies = []
    mus = []
    theta = np.zeros(env.n_states)

    for it in range(n_iters):
        if len(mus) == 0:
            # Start with uniform random policy
            pol = np.ones((env.n_states, env.n_actions)) / env.n_actions
            policies.append(pol)
            mus.append(compute_policy_mu(env, mdp, pol))
            continue

        # Build matrix of differences
        diffs = np.array([mu_expert - mu for mu in mus])  # shape (m, S)

        if CPX_AVAILABLE:
            # CVXPY QP: maximize t subject to theta^T diffs[j] >= t for all j, ||theta||_2 <= 1
            m = diffs.shape[0]
            theta_var = cp.Variable(env.n_states)
            t = cp.Variable()
            constraints = [cp.norm(theta_var, 2) <= 1, t >= 0]
            for j in range(m):
                constraints.append(theta_var @ diffs[j] >= t)
            prob = cp.Problem(cp.Maximize(t), constraints)
            prob.solve(solver=cp.SCS, verbose=False)
            if prob.status != cp.OPTIMAL and prob.status != cp.OPTIMAL_INACCURATE:
                print(f"CVXPY solver status: {prob.status}; falling back to PGD for this iter")
                CPX_OK = False
            else:
                theta = theta_var.value
        else:
            # Projected subgradient method to maximize the minimum margin
            theta = np.random.randn(env.n_states) * 0.01 if np.all(theta == 0) else theta
            lr = 0.5
            for _ in range(200):
                margins = diffs @ theta  # shape (m,)
                j_min = np.argmin(margins)
                grad = diffs[j_min]
                theta += lr * grad
                theta = project_to_l2_ball(theta, radius=1.0)
                if margins[j_min] >= epsilon:
                    break

        # Derive policy by value iteration on reward theta^T phi(s) (phi(s) = one-hot -> reward = theta)
        learned_r = theta.copy()
        new_policy = value_iteration(mdp, learned_r)
        # convert to deterministic policy matrix
        pol_mat = np.zeros((env.n_states, env.n_actions))
        pol_mat[np.arange(env.n_states), new_policy] = 1.0
        policies.append(pol_mat)
        mu_new = compute_policy_mu(env, mdp, pol_mat)
        mus.append(mu_new)

        # stopping condition: check margin t
        margins = np.array([theta.dot(mu_expert - mu) for mu in mus])
        t_margin = margins.min()
        print(f"Apprenticeship iter {it+1}: margin={t_margin:.4f}, num_policies={len(mus)}")
        if t_margin >= epsilon:
            break

    return theta, policies

# run apprenticeship learning
theta_appr, policies_appr = apprenticeship_learning(env, mdp, mu_expert, n_iters=20, epsilon=0.01)
env.render_reward(theta_appr, title='Apprenticeship Recovered Reward')
match_appr = evaluate_policy_match(env, mdp, expert_policy, theta_appr)
print(f"Apprenticeship policy match: {match_appr:.3f}")

In [ ]:
# -----------------------------
# 5b. Max-Margin IRL (primal approximation via hinge loss)
# -----------------------------

def max_margin_irl_primal(env: GridWorld, mdp: MDP, mu_expert: np.ndarray, n_iters: int = 200, C: float = 1.0, lr: float = 0.1) -> np.ndarray:
    """Approximate max-margin solution via primal hinge loss with L2 regularization.
    Minimizes: 0.5 * ||theta||^2 + C * sum_j max(0, 1 - theta^T (mu_expert - mu_j))
    Requires a set of learner feature expectations mu_j; we build them iteratively by alternating policy updates (similar to apprenticeship learning).
    """
    theta = np.zeros(env.n_states)
    mus = []
    for it in range(n_iters):
        # compute current reward and greedy policy
        rewards = theta.copy()
        policy = value_iteration(mdp, rewards)
        pol_mat = np.zeros((env.n_states, env.n_actions))
        pol_mat[np.arange(env.n_states), policy] = 1.0
        mu_j = compute_policy_mu(env, mdp, pol_mat)
        mus.append(mu_j)

        # compute hinge loss gradients over collected mus
        diffs = np.array([mu_expert - mu for mu in mus])
        gaps = 1.0 - (diffs @ theta)  # vector of size m
        hinge_mask = (gaps > 0).astype(float)
        grad = theta - C * (hinge_mask[:, None] * diffs).sum(axis=0)
        theta = theta - lr * grad
        if it % max(1, n_iters // 10) == 0:
            loss = 0.5 * np.dot(theta, theta) + C * np.maximum(0, gaps).sum()
            print(f"Iter {it+1}/{n_iters}: loss={loss:.6f}, num_mus={len(mus)}")
    return theta

# run approximate max-margin
theta_mm_approx = max_margin_irl_primal(env, mdp, mu_expert, n_iters=120, C=1.0, lr=0.1)
env.render_reward(theta_mm_approx, title='Max-Margin (primal approx) Recovered Reward')
match_mm_approx = evaluate_policy_match(env, mdp, expert_policy, theta_mm_approx)
print(f"Max-Margin (approx) policy match: {match_mm_approx:.3f}")

## 3) Ground-truth reward & expert trajectories
We define a simple ground truth reward with a goal (positive) and an obstacle (negative), then compute an optimal deterministic policy via value iteration.

In [ ]:
def value_iteration(env: GridWorld, reward: np.ndarray, gamma: float = 0.9, tol: float = 1e-6, max_iters: int = 1000) -> np.ndarray:
    V = np.zeros(env.n_states, dtype=float)
    for _ in range(max_iters):
        Q = reward[:, None] + gamma * (env.P @ V)
        V_new = np.max(Q, axis=1)
        if np.max(np.abs(V - V_new)) < tol:
            break
        V = V_new
    # deterministic greedy policy
    Q = reward[:, None] + gamma * (env.P @ V)
    policy = np.argmax(Q, axis=1)
    return policy


def generate_expert_trajectories(env: GridWorld, policy: np.ndarray, n_trajs: int = 50, len_traj: int = 20, start_states: Optional[List[int]] = None) -> List[List[Tuple[int,int]]]:
    trajectories = []
    if start_states is None:
        start_states = list(range(env.n_states))
    for _ in range(n_trajs):
        s = np.random.choice(start_states)
        traj = []
        for _ in range(len_traj):
            a = policy[s]
            traj.append((s, int(a)))
            s = env._get_next_state(s, a)
            if s == env.goal_state:
                break
        trajectories.append(traj)
    return trajectories

# Build a GT reward
gt_rewards = np.full(env.n_states, -0.1)
gt_rewards[env.goal_state] = 1.0
# Put a fire pit in the middle if grid is odd-sized
mid = (env.size // 2) * env.size + (env.size // 2)
gt_rewards[mid] = -1.0

expert_policy = value_iteration(env, gt_rewards, gamma=0.9)
demonstrations = generate_expert_trajectories(env, expert_policy, n_trajs=80, len_traj=25)
print(f"Generated {len(demonstrations)} expert trajectories. Avg length ~{np.mean([len(t) for t in demonstrations]):.2f} steps")

# Visualize GT reward
env.render_reward(gt_rewards, title='Ground Truth Reward')

## 4) Expert feature expectations (mu_E)
We compute the discounted empirical feature counts from demonstrations: mu_E = E[ sum_t gamma^t phi(s_t) ]

In [ ]:
def compute_expert_feature_expectations(env: GridWorld, trajectories: List[List[Tuple[int,int]]], gamma: float = 0.9) -> np.ndarray:
    mu = np.zeros(env.n_states, dtype=float)
    for traj in trajectories:
        for t, (s, _) in enumerate(traj):
            mu += (gamma ** t) * env.get_features(s)
    mu /= len(trajectories)
    return mu

mu_expert = compute_expert_feature_expectations(env, demonstrations, gamma=0.9)
print("Computed expert feature expectations (mu_expert). Sum of discounted counts:", mu_expert.sum())

## 5) MaxEnt IRL — Implementation details and improved numerics
Key improvements:
- Use temperature scaling (beta) to control softness
- Use log-sum-exp (stable) for soft value iteration
- Use expert start-state distribution instead of uniform
- L2 regularization on theta

In [ ]:
def soft_value_iteration(env: GridWorld, rewards: np.ndarray, gamma: float = 0.9, beta: float = 1.0, tol: float = 1e-6, max_iters: int = 200) -> np.ndarray:
    """Compute soft state values V(s) via iterative soft Bellman updates.
    V(s) = logsumexp( (R(s) + gamma * Sum_s' P(s'|s,a) V(s')) * beta ) / beta
    We operate in log-space for numerical stability.
    """
    n = env.n_states
    V = np.zeros(n, dtype=float)
    for _ in range(max_iters):
        Q = rewards[:, None] + gamma * (env.P @ V)  # shape (S, A)
        # apply temperature beta: V = (1/beta) * logsumexp(beta * Q, axis=1)
        V_new = logsumexp(beta * Q, axis=1) / max(1e-12, beta)
        if np.max(np.abs(V - V_new)) < tol:
            V = V_new
            break
        V = V_new
    return V


def compute_policy_from_V(env: GridWorld, rewards: np.ndarray, V: np.ndarray, gamma: float = 0.9, beta: float = 1.0) -> np.ndarray:
    Q = rewards[:, None] + gamma * (env.P @ V)
    # soft policy pi(a|s) proportional to exp(beta * (Q - V[:,None]))
    logits = beta * (Q - V[:, None])
    # subtract max for stability
    logits = logits - np.max(logits, axis=1, keepdims=True)
    pi = np.exp(logits)
    pi = pi / (pi.sum(axis=1, keepdims=True) + 1e-12)
    return pi


def compute_state_visitation_freq(env: GridWorld, policy: np.ndarray, start_dist: np.ndarray, gamma: float = 0.9, horizon: int = 50) -> np.ndarray:
    """Forward pass: compute discounted state visitation frequencies D(s) under soft policy.
    policy: (S, A) probabilities
    start_dist: initial state distribution
    Returns discounted cumulative visitation D_cum(s) = sum_t gamma^t d_t(s)
    """
    n = env.n_states
    d = start_dist.copy()
    d_cum = np.zeros(n, dtype=float)
    for t in range(horizon):
        d_cum += (gamma ** t) * d
        # next-state distribution
        # T[s, s'] = sum_a pi(a|s) * P[s, a, s']
        T = np.sum(env.P * policy[:, :, None], axis=1)  # shape (S, S)
        d = d @ T
    return d_cum


def max_ent_irl(env: GridWorld, mu_expert: np.ndarray, n_iters: int = 200, lr: float = 0.2, gamma: float = 0.9, beta: float = 1.0, l2: float = 1e-3, start_dist: Optional[np.ndarray] = None, verbose: bool = True) -> Tuple[np.ndarray, dict]:
    """Learn reward weights theta (one per state) by MaxEnt IRL.
    Returns theta and diagnostics dict.
    """
    S = env.n_states
    theta = np.random.randn(S) * 0.01
    phi = env.feature_matrix()  # identity

    if start_dist is None:
        start_dist = np.ones(S) / S

    loss_hist = []
    grad_norms = []

    for i in range(n_iters):
        rewards = phi @ theta
        V = soft_value_iteration(env, rewards, gamma=gamma, beta=beta)
        pi = compute_policy_from_V(env, rewards, V, gamma=gamma, beta=beta)
        mu_model = compute_state_visitation_freq(env, pi, start_dist, gamma=gamma)

        grad = mu_expert - mu_model - l2 * theta  # with L2 regularization
        theta += lr * grad

        gnorm = np.linalg.norm(grad)
        loss_hist.append(float(gnorm))
        grad_norms.append(float(gnorm))

        if verbose and (i % max(10, n_iters//10) == 0 or i == n_iters-1):
            print(f"Iter {i+1}/{n_iters}: grad_norm={gnorm:.6f}")

    diagnostics = {
        'loss_hist': loss_hist,
        'theta': theta
    }
    return theta, diagnostics

## 6) Run IRL and recover reward
We'll run the MaxEnt IRL algorithm and then evaluate how well the recovered reward induces a policy that matches the expert.

In [ ]:
# Build start distribution from demonstrations (empirical start states)
start_counts = np.zeros(env.n_states, dtype=float)
for traj in demonstrations:
    start_counts[traj[0][0]] += 1
start_dist_emp = start_counts / start_counts.sum()

theta_learned, diag = max_ent_irl(env, mu_expert, n_iters=200, lr=0.25, gamma=0.9, beta=1.0, l2=1e-3, start_dist=start_dist_emp, verbose=True)
recovered_rewards = env.feature_matrix() @ theta_learned

# Visualize recovered rewards (normalized)
recovered_norm = (recovered_rewards - recovered_rewards.mean()) / (recovered_rewards.std() + 1e-12)
env.render_reward(gt_rewards.reshape(-1), title='Ground Truth Reward')
env.render_reward(recovered_norm, title='Recovered Reward (normalized)')

# Plot loss curve
plt.figure(figsize=(6,3)); plt.plot(diag['loss_hist']); plt.title('Gradient Norm (convergence proxy)'); plt.xlabel('Iteration'); plt.show()

## 7) Evaluation metrics
- Policy alignment: fraction of states where greedy policy under recovered reward matches expert policy
- Return comparison: average episodic return of learned greedy policy vs expert

In [ ]:
def greedy_policy_from_rewards(env: GridWorld, rewards: np.ndarray, gamma: float = 0.9) -> np.ndarray:
    Q = rewards[:, None] + gamma * (env.P @ np.zeros(env.n_states))
    # One-step greedy wrt immediate reward + next-state value approximated by zero (but better: run value iteration to get proper greedy policy)
    # Use full value iteration to be consistent
    return value_iteration(env, rewards, gamma=gamma)

# Expert policy vs recovered policy match
recovered_policy = greedy_policy_from_rewards(env, recovered_rewards, gamma=0.9)
match_frac = (recovered_policy == expert_policy).mean()
print(f"Policy match fraction (greedy): {match_frac:.3f}")

# Evaluate average return by rolling out policies (stochastic env accounted for in env.P)
def rollout_policy(env: GridWorld, policy: np.ndarray, n_episodes: int = 200, max_steps: int = 100, reward_fn: Optional[np.ndarray] = None) -> float:
    if reward_fn is None:
        reward_fn = gt_rewards
    returns = []
    for _ in range(n_episodes):
        s = np.random.choice(env.n_states, p=start_dist_emp)
        total = 0.0
        for _ in range(max_steps):
            a = policy[s]
            # sample next state according to P
            ns = np.random.choice(env.n_states, p=env.P[s, a])
            total += reward_fn[ns]
            s = ns
            if s == env.goal_state:
                break
        returns.append(total)
    return np.mean(returns)

expert_return = rollout_policy(env, expert_policy, n_episodes=200)
recovered_return = rollout_policy(env, recovered_policy, n_episodes=200)
print(f"Expert avg return: {expert_return:.3f} | Recovered greedy avg return: {recovered_return:.3f}")

## 8) Extra diagnostics & visualizations
- Histogram of recovered reward values
- Per-state difference in state visitation frequencies (mu_expert vs mu_model)

In [ ]:
# Compute mu_model from final learned reward
V_final = soft_value_iteration(env, recovered_rewards, gamma=0.9, beta=1.0)
pi_final = compute_policy_from_V(env, recovered_rewards, V_final, gamma=0.9, beta=1.0)
mu_model = compute_state_visitation_freq(env, pi_final, start_dist_emp, gamma=0.9)

plt.figure(figsize=(10,4))
plt.subplot(1,3,1)
sns.histplot(recovered_rewards, bins=10, kde=True); plt.title('Recovered Reward distribution')

plt.subplot(1,3,2)
plt.imshow((mu_expert - mu_model).reshape(env.size, env.size), cmap='bwr'); plt.colorbar(); plt.title('mu_expert - mu_model')

plt.subplot(1,3,3)
plt.plot(diag['loss_hist']); plt.title('Gradient norm')
plt.tight_layout(); plt.show()

print(f"Sum mu_expert: {mu_expert.sum():.3f} | Sum mu_model: {mu_model.sum():.3f}")

## Lecture 23 Connections — Equations & Key Points

This notebook implemented the major concepts from Lecture 23. For convenience, here are the central equations and their implementations:

- Value and Q-functions (Eq 1.2): V^π_R(s) = E_π[∑ γ^t R(s_t,a_t)] and Q^π_R(s,a) = R(s,a) + γ E_{s'|s,a}[V^π_R(s')]. Implemented in `value_iteration` and `compute_policy_mu` utilities.

- Feature expectations (Eq 3.2): μ_E = (1/N) ∑_i ∑_t γ^t φ(s_t,a_t); implemented as `compute_expert_feature_expectations` (and `compute_expert_mu`).

- MaxEnt objective (Eq 6.4 & 6.5): P_θ(τ) ∝ exp(∑_t R_θ(s_t,a_t)). Partition function log Z(θ) = V_θ(s_0) computed by soft value iteration (Eq 7.3). Implemented in `soft_value_iteration` and `max_ent_irl`.

- Soft Bellman (Eq 7.1): V(s) = log ∑_a exp(Q(s,a)) implemented via numerically-stable `logsumexp` in `soft_value_iteration`.

- Gradient of log-likelihood (Eq 7.5): ∇_θ log P_θ(D) = N (μ_E − μ_{P_θ}). Implemented as the driving gradient in `max_ent_irl` and in sample-based and model-free variants.

- Importance sampling estimator (Eq 8.1) and ESS diagnostic (Sec 9): implemented in `sample_based_maxent` and `estimate_mu_from_weighted_trajectories`.

- GCL/GAIL: Adversarial training where the reward/classifier is trained to discriminate expert vs learner trajectories while the policy is updated to minimize the learned cost. Implemented as `train_gail` (REINFORCE updates). For production, using TRPO/PPO with a KL/Trust region or stable library is recommended.

If you'd like, I can add LaTeX-rendered equations inside the notebook markdown cells for clearer presentation, or export a PDF summary that matches the lecture notes (with references to the equations above).

## 9) Quick ablation experiments (try different temperature / regularization)
You can vary `beta` (temperature), `l2` and `lr` to see their effects on stability and match.

In [ ]:
# Quick grid search over beta values to show sensitivity
betas = [0.5, 1.0, 2.0]
results = {}
for b in betas:
    theta_b, diag_b = max_ent_irl(env, mu_expert, n_iters=120, lr=0.2, gamma=0.9, beta=b, l2=1e-3, start_dist=start_dist_emp, verbose=False)
    rec_r = env.feature_matrix() @ theta_b
    rec_pol = greedy_policy_from_rewards(env, rec_r, gamma=0.9)
    match = (rec_pol == expert_policy).mean()
    results[b] = match

print("Beta -> policy match fraction:")
for b, m in results.items(): print(f"  {b}: {m:.3f}")

## 10) Conclusions & next steps ✅
- MaxEnt IRL reliably recovers reward structure that yields policies similar to the expert (within invariances).
- Practical improvements used here: start-state matching, L2 regularization, temperature parameter (beta), and numerically-stable soft value iteration.

Next suggestions:
- Replace linear features with learned features (CNN or random features) to scale to larger state spaces.
- Implement baselines (e.g., feature-matching QP) for comparison.
- Add GPU-accelerated batch computations for very large problems.

---

If you'd like, I can:
- Run the notebook locally and tune hyperparameters for you, or
- Convert this into a short script with CLI options and unit tests.

Tell me which you'd prefer. ✅